 # 0.03 — training_features Audit (sklearn readiness)

 Answers one question: is `training_features` clean enough to fit models on?

 Six checks, in order of severity:
 1. **Target health** — NULL targets poison metrics; must be 0.
 2. **NaN / NULL per feature** — linear models throw on NaN; XGBoost tolerates it.
    All NaNs here are *legitimate* (lags at hourly gaps, weather gaps, structural
    proximity NULLs) — the audit just maps them so the trainer knows what to impute.
 3. **inf / -inf** — silently break StandardScaler and explode Ridge.
 4. **Constant columns** — zero-variance breaks per-fold scaling (÷0), adds nothing.
 5. **Impossible negatives** — negative bike/dock counts are parse or sensor glitches.
 6. **Non-numeric columns** — `season`, `station_role` need encoding before any
    numeric model; booleans need explicit casting.

 Authored as a `# %%` .py file (clean git diffs). Export to `.ipynb` with outputs via:
   Command Palette → "Jupyter: Export Current Python File as Jupyter Notebook"

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import psycopg2

# Make citibike package importable when run from notebooks/ or project root.
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from citibike.config import DB_CONFIG  # noqa: E402
from model_training.build_training_features_pandas import (  # noqa: E402
    INSERT_COLUMNS, INT64_COLUMNS, BOOL_COLUMNS,
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

 ## Configuration
 Change `TABLE` to audit a different table (e.g. `training_features_pandas`).

In [ ]:
TABLE = "training_features"

PK      = ["station_id", "timestamp", "horizon_minutes"]
TARGETS = ["bikes_available_at_horizon", "bike_available_binary"]
TEXT_COLS  = ["season", "station_role"]
NONNEG_INT = [
    "num_bikes_available", "num_ebikes_available", "num_docks_available",
    "num_bikes_disabled", "capacity", "departures_this_hour", "arrivals_this_hour",
]

feature_cols  = [c for c in INSERT_COLUMNS if c not in PK and c not in TARGETS]
numeric_feats = [c for c in feature_cols if c not in TEXT_COLS and c not in BOOL_COLUMNS]
float_feats   = [c for c in numeric_feats if c not in INT64_COLUMNS]

def q(col):
    """Quote timestamp column name for SQL; leave others bare."""
    return f'"{col}"' if col == "timestamp" else col

conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

cur.execute(f"SELECT count(*) FROM {TABLE};")
n_rows = cur.fetchone()[0]
print(f"TABLE {TABLE}: {n_rows:,} rows")

TABLE training_features: 161,331,182 rows


 ## 1. Target health
 Both targets must have zero NULLs. The feature builder drops NULL-target rows
 before writing, so any NULLs here indicate a builder bug.

In [ ]:
rows = []
for tg in TARGETS:
    cur.execute(f"SELECT count(*) FROM {TABLE} WHERE {tg} IS NULL;")
    nn = cur.fetchone()[0]
    rows.append({"target": tg, "null_count": nn, "status": "✓ ok" if nn == 0 else "*** NULL TARGETS ***"})

pd.DataFrame(rows).set_index("target")

,null_count,status
target,,
bikes_available_at_horizon,0,✓ ok
bike_available_binary,0,✓ ok


 ## 2. NULL / NaN per feature column
 All NaNs listed here are *expected*:
 - Lag columns (`bikes_1hr_ago` etc.) — NaN at hourly gaps in the clean table.
 - Weather columns — coverage gaps; filled on next ingest run.
 - `nearest_entrance_dist_m` ~15% — structural (no subway within 800m); use a
   sentinel / cap, never statistically impute (see CLAUDE.md Section B).
 - `rate_of_change_*` — 100% NULL; deferred sub-hourly feature; drop before training.

 XGBoost trains as-is. Linear / Ridge / Logistic need `SimpleImputer(median)`
 inside a per-fold `Pipeline` for the non-structural NaNs.

In [ ]:
null_exprs = ", ".join(
    f"SUM(CASE WHEN {q(c)} IS NULL THEN 1 ELSE 0 END) AS {c.replace('.', '_')}"
    for c in feature_cols
)
cur.execute(f"SELECT {null_exprs} FROM {TABLE};")
raw = dict(zip(feature_cols, cur.fetchone()))

null_df = (
    pd.DataFrame([{"feature": c, "null_count": v or 0, "pct": 100.0 * (v or 0) / n_rows}
                  for c, v in raw.items()])
    .query("null_count > 0")
    .sort_values("null_count", ascending=False)
    .reset_index(drop=True)
)
null_df["null_count"] = null_df["null_count"].map("{:,}".format)
null_df["pct"] = null_df["pct"].map("{:.2f}%".format)
null_df

,feature,null_count,pct
0,rate_of_change_30min,"161,331,182",100.00%
1,rate_of_change_20min,"161,331,182",100.00%
2,rate_of_change_10min,"161,331,182",100.00%
3,nearest_entrance_dist_m,"149,685,859",92.78%
4,is_within_400m,"147,538,162",91.45%
5,entrance_count_800m,"147,538,162",91.45%
6,entrance_count_400m,"147,538,162",91.45%
7,change_classic_12hr,"112,271,597",69.59%
8,change_ebikes_12hr,"112,271,597",69.59%
9,change_ebikes_6hr,"111,503,174",69.11%


 ## 3. inf / -inf in float columns
 Any row here breaks `StandardScaler` and causes Ridge to diverge.
 They come from divide-by-zero in ratio features (e.g. `fill_ratio` when
 `capacity = 0`). Should be none — the builder masks zero capacity to NaN first.

In [ ]:
inf_exprs = ", ".join(
    f"SUM(CASE WHEN {c} = 'Infinity'::float8 OR {c} = '-Infinity'::float8 "
    f"THEN 1 ELSE 0 END) AS {c.replace('.', '_')}"
    for c in float_feats
)
cur.execute(f"SELECT {inf_exprs} FROM {TABLE};")
inf_raw = dict(zip(float_feats, cur.fetchone()))

inf_df = (
    pd.DataFrame([{"feature": c, "inf_count": v or 0} for c, v in inf_raw.items()])
    .query("inf_count > 0")
    .reset_index(drop=True)
)
if inf_df.empty:
    print("(none) — no inf / -inf values found")
else:
    inf_df

(none) — no inf / -inf values found


 ## 4. Constant (zero-variance) columns
 A column with the same value in every row adds no signal and breaks
 `StandardScaler` (division by zero in the variance). Drop before training.
 Note: `snowfall` will appear constant on a summer-only window — check again
 after the full historical backfill covers winter months.

In [ ]:
const_rows = []
for c in numeric_feats:
    cur.execute(f"SELECT min({c}), max({c}), count({c}) FROM {TABLE};")
    mn, mx, nonnull = cur.fetchone()
    if nonnull and mn is not None and mn == mx:
        const_rows.append({"feature": c, "constant_value": mn})

const_df = pd.DataFrame(const_rows)
if const_df.empty:
    print("(none) — no constant columns")
else:
    const_df

(none) — no constant columns


 ## 5. Impossible values (negative counts)
 Negative bike / dock counts are impossible and indicate a parse glitch or
 Kaggle CSV artifact. Any flagged here should be investigated and the cleaning
 rule added to `clean_month()` in `build_clean_availability.py`.

In [ ]:
neg_rows = []
for c in NONNEG_INT:
    cur.execute(f"SELECT count(*) FROM {TABLE} WHERE {c} < 0;")
    nn = cur.fetchone()[0]
    if nn:
        neg_rows.append({"feature": c, "negative_count": nn})

neg_df = pd.DataFrame(neg_rows)
if neg_df.empty:
    print("(none) — no negative counts")
else:
    neg_df

(none) — no negative counts


 ## 6. Non-numeric feature columns
 These columns need encoding before any numeric model:
 - `season`, `station_role` → one-hot or ordinal encode
 - `is_weekend`, `is_holiday`, `is_within_400m` → cast to int (already 0/1,
   but pandas `boolean` dtype needs explicit conversion for sklearn)

In [ ]:
print("Text columns (one-hot or ordinal encode before numeric model):")
for c in TEXT_COLS:
    cur.execute(f"SELECT {c}, count(*) FROM {TABLE} GROUP BY {c} ORDER BY count(*) DESC;")
    vals = cur.fetchall()
    print(f"\n  {c}:")
    for v, n in vals:
        print(f"    {str(v):<20} {n:>12,} rows  ({100.0*n/n_rows:.1f}%)")

print(f"\nBoolean columns (cast to int before sklearn):")
for c in BOOL_COLUMNS:
    print(f"  {c}")

Text columns (one-hot or ordinal encode before numeric model):

  season:
    spring                 46,650,144 rows  (28.9%)
    fall                   43,239,557 rows  (26.8%)
    summer                 38,184,628 rows  (23.7%)
    winter                 33,256,853 rows  (20.6%)

  station_role:
    balanced              134,163,331 rows  (83.2%)
    None                   15,560,940 rows  (9.6%)
    sink                    9,096,847 rows  (5.6%)
    source                  2,510,064 rows  (1.6%)

Boolean columns (cast to int before sklearn):
  is_weekend
  is_holiday
  is_within_400m


 ## Summary
 - **Targets**: must be 0 NULLs (check section 1)
 - **XGBoost**: train as-is — handles NaN natively, scale-invariant
 - **Linear / Ridge / Logistic**: wrap in `Pipeline(SimpleImputer → StandardScaler)`
   fit inside each CV fold; never fit the imputer on the full dataset
 - **Drop before any model**: `rate_of_change_10/20/30min` (100% NULL), any
   constant columns from section 4
 - **Encode before numeric model**: `season`, `station_role` (section 6)
 - **Do NOT impute**: `nearest_entrance_dist_m` structural NULLs — use a
   sentinel value (e.g. 800) instead; the signal is already in `is_within_400m`

In [ ]:
conn.close()
print("Audit complete.")

Audit complete.
